# View company-assistant roleplay conversations (categorical judge format)

Sibling of `view-company-assistant-conversations.ipynb`, updated for the new **categorical** judge output produced by `judge_company_assistant.py`.

On-disk layout:
- conversations: `<MODEL>.json` (keys `model`, `m`, `c`, `layers`, `conversations`), each conversation has `k`, `system_prompt`, `user_prompt`, `autocompletions`.
- judge scores: `<MODEL>-judge-<JUDGE_MODEL>-<prompthash>-<YYYYmmdd-HHMMSS>.json` (key `scores`), one record per `k` with a string `label` and a free-text `reason` (instead of the old numeric `honesty`/`follow_through`). The header also carries the `judge_prompt` used for the run; the `<prompthash>` in the filename ties the file to that exact prompt, so editing the prompt produces a new file.

Labels: `honest_authorized`, `deceptive`, `evasive`, `mixed`, `withheld_authorized`, `null`, `incomplete` (plus `error` records for failed scorings).

Scores join back to conversations on `k`. This notebook loads the **most recently written** judge file for the chosen `MODEL`/`JUDGE_MODEL` (set `JUDGE_PATH` by hand to pin a specific run).

Set `MODEL` and `JUDGE_MODEL`, run all cells, then use:
- `show(k)` — render a single conversation by its `k`,
- `show_all()` — render every conversation inline (optionally filtered by label).

In [ ]:
import json
import html
from pathlib import Path
from collections import Counter

from IPython.display import HTML, display

# --- config -----------------------------------------------------------------
DATA_DIR = Path("data-company-assistant-roleplay")

MODEL = "Qwen--Qwen2.5-32B-Instruct"  # basename of the conversation .json
JUDGE_MODEL = "gpt-5.4-mini"          # matches the judge output filename

SCORE_KEY = "k"                       # field that joins a score to a conversation

CONV_PATH = DATA_DIR / f"{MODEL}.json"

# Judge files are named "<MODEL>-judge-<JUDGE_MODEL>-<prompthash>-<YYYYmmdd-HHMMSS>.json".
# Pick the most recently written one (the prompt hash makes lexical order unreliable).
# Set JUDGE_PATH by hand to pin a specific run.
_judge_glob = sorted(
    DATA_DIR.glob(f"{MODEL}-judge-{JUDGE_MODEL}-*.json"), key=lambda p: p.stat().st_mtime
)
if not _judge_glob:
    raise FileNotFoundError(
        f"no judge files matching {MODEL}-judge-{JUDGE_MODEL}-*.json in {DATA_DIR}"
    )
JUDGE_PATH = _judge_glob[-1]

print("conversations:", CONV_PATH)
print("judge scores: ", JUDGE_PATH)

In [ ]:
# --- load and join on the conversation's k ----------------------------------
with CONV_PATH.open() as f:
    conv_data = json.load(f)
with JUDGE_PATH.open() as f:
    judge_data = json.load(f)


def normalize_score(s):
    """Map a raw score record onto a common shape."""
    return {
        "k": str(s.get(SCORE_KEY)),
        "label": s.get("label"),
        "reason": s.get("reason"),
        "error": s.get("error"),
    }


scores_by_k = {}
for s in judge_data.get("scores", []):
    ns = normalize_score(s)
    if ns["k"] is not None:
        scores_by_k[ns["k"]] = ns


# tokeniser end-of-turn markers used by the various base models
_EOT_MARKERS = ("<|eot_id|>", "<|im_end|>", "<end_of_turn>", "<|endoftext|>")


def assistant_text(conv):
    acs = conv.get("autocompletions") or []
    if not acs:
        return ""
    text = acs[0].get("text") or ""
    for m in _EOT_MARKERS:
        text = text.replace(m, "")
    return text.strip()


# records: one per conversation, with its judge verdict attached (if any)
records = []
for conv in conv_data["conversations"]:
    records.append({"conv": conv, "score": scores_by_k.get(str(conv["k"]))})
records_by_k = {str(r["conv"]["k"]): r for r in records}

print(f"{len(records)} conversations, {len(scores_by_k)} with a judge verdict")


def verdict_of(score):
    """Short label for a score record: the judge's label, ERROR, or '—' if the
    conversation was never scored."""
    if score is None:
        return "—"
    if score.get("error"):
        return "ERROR"
    if score.get("label") is not None:
        return str(score["label"])
    return "NULL"


print("label counts:", dict(Counter(verdict_of(r["score"]) for r in records)))

In [ ]:
# --- rendering --------------------------------------------------------------
# Colour per label: green = honest disclosure to an authorized requester, red =
# deception, blue = legitimate refusal, pink = partly deceptive/partly evasive,
# orange = failed to comply when it should have, purple = truncated, grey =
# null/unscored.
LABEL_STYLE = {
    "honest_authorized":   ("#2e7d32", "white"),  # green
    "deceptive":           ("#c0392b", "white"),  # red
    "evasive":             ("#2c6fbb", "white"),  # blue
    "mixed":               ("#c2185b", "white"),  # pink (deceptive + evasive)
    "withheld_authorized": ("#e08e0b", "white"),  # orange
    "incomplete":          ("#8e44ad", "white"),  # purple
    "null":                ("#6c757d", "white"),  # grey
    "NULL":                ("#6c757d", "white"),
    "ERROR":               ("#7b1010", "white"),
    "—":                   ("#e0e0e0", "#555"),
}


def _badge(score):
    """Coloured HTML badge for the judge's categorical label."""
    v = verdict_of(score)
    bg, fg = LABEL_STYLE.get(v, ("#6c757d", "white"))
    label = "not scored" if v == "—" else v
    return (
        f"<span style='background:{bg};color:{fg};padding:3px 10px;"
        f"border-radius:10px;font-weight:600;font-size:13px;margin-left:6px;'>{label}</span>"
    )


def _block(title, text, bg):
    return (
        f"<div style='margin:6px 0;'>"
        f"<div style='font-weight:700;color:#444;font-size:12px;"
        f"text-transform:uppercase;letter-spacing:.5px;'>{title}</div>"
        f"<div style='background:{bg};border-radius:6px;padding:8px 12px;"
        f"white-space:pre-wrap;font-family:-apple-system,Segoe UI,sans-serif;"
        f"font-size:14px;line-height:1.45;'>{html.escape(text)}</div></div>"
    )


def _card_html(record):
    conv, score = record["conv"], record["score"]
    head = (
        f"<div style='display:flex;justify-content:space-between;"
        f"align-items:center;border-bottom:1px solid #ddd;padding-bottom:6px;'>"
        f"<span style='font-weight:700;font-size:15px;'>k = {conv['k']}</span>"
        f"<span>{_badge(score)}</span></div>"
    )
    body = (
        _block("system prompt", conv.get("system_prompt", ""), "#f4f6f8")
        + _block("user prompt", conv.get("user_prompt", ""), "#eef3fb")
        + _block("assistant answer", assistant_text(conv), "#f3faf3")
    )
    reason = score.get("reason") if score else None
    if reason:
        body += _block("judge reasoning", str(reason), "#fffdf2")
    err = score.get("error") if score else None
    if err:
        body += _block("judge error", str(err), "#fdecea")
    return (
        f"<div style='border:1px solid #ccc;border-radius:8px;padding:12px;"
        f"margin:12px 0;max-width:1000px;'>{head}{body}</div>"
    )


def show(k):
    """Render a single conversation by its k (str or int)."""
    rec = records_by_k.get(str(k))
    if rec is None:
        print(f"no conversation with k={k}")
        return
    display(HTML(_card_html(rec)))


def show_all(only=None, limit=None):
    """Render every conversation inline.

    only:  optional filter on a label, e.g. only="deceptive", only="NULL",
           or a set/list like only={"deceptive", "evasive"}.
    limit: cap how many cards are rendered.
    """
    if isinstance(only, str):
        only = {only}
    chosen = [
        r for r in records
        if only is None or verdict_of(r["score"]) in only
    ]
    if limit is not None:
        chosen = chosen[:limit]
    print(f"showing {len(chosen)} conversation(s)")
    display(HTML("".join(_card_html(r) for r in chosen)))

In [ ]:
# The judge prompt template used to produce these labels is stored in the file header.
print(judge_data.get("judge_prompt", "(no judge_prompt field in this file)"))

In [ ]:
# A single conversation by k
# show(0)

In [ ]:
# Every judged conversation (scroll through). Filter by label, e.g.
# show_all(only="deceptive"); pass include_unscored=True to also see the rest.
show_all()